# 📊 Notebook 03 — Exploratory Data Analysis (EDA)
## Spatio-Temporal PM2.5/PM10 Analysis — Karachi, Pakistan

**This notebook:**
1. Dataset overview — shape, dtypes, missing values
2. PM2.5 distribution analysis — histograms, KDE, QQ-plots
3. Temporal trends — annual, monthly, seasonal, day-of-week
4. Station-level comparison
5. Meteorological correlations with PM2.5
6. Satellite feature analysis (AOD, AER-AI, VIIRS NTL)
7. WHO exceedance analysis
8. Correlation heatmap

**Input:** `data/processed/master_dataset.csv`

## 0. Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

Path('outputs').mkdir(exist_ok=True)

plt.rcParams.update({
    'figure.facecolor': '#0d0d14', 'axes.facecolor': '#111118',
    'axes.edgecolor': '#222233',   'axes.labelcolor': '#aaaacc',
    'xtick.color': '#666688',      'ytick.color': '#666688',
    'text.color': '#e8e8f0',       'grid.color': '#1a1a2a',
    'grid.linestyle': '--',        'grid.alpha': 0.5,
    'font.family': 'monospace',
})
PALETTE = ['#c8f04a','#4af0c8','#f04a7a','#f0c84a','#7a4af0','#4a7af0','#f07a4a','#4af07a']
WHO_ANNUAL  = 5    # µg/m³ annual guideline
WHO_24H     = 15   # µg/m³ 24-hour guideline
print('✓ Imports loaded')

✓ Imports loaded


## 1. Load Dataset & Overview

In [ ]:
df = pd.read_csv('../data/processed/master_dataset.csv')
df['date'] = pd.to_datetime(df['date'])

# Fix temperature if still in Kelvin
if 'temperature_2m' in df.columns and df['temperature_2m'].mean() > 100:
    df['temperature_2m'] = df['temperature_2m'] - 273.15
if 'temp_c' not in df.columns and 'temperature_2m' in df.columns:
    df['temp_c'] = df['temperature_2m']

print('=' * 60)
print('DATASET OVERVIEW')
print('=' * 60)
print(f'  Shape         : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'  Date range    : {df["date"].min().date()} → {df["date"].max().date()}')
print(f'  Stations      : {df["station"].nunique()} — {df["station"].unique().tolist()}')
print(f'  Years covered : {df["date"].dt.year.nunique()} ({df["date"].dt.year.min()}–{df["date"].dt.year.max()})')
print()
print('PM2.5 Summary Statistics:')
print(df['pm25'].describe().round(2).to_string())
print()
miss = df.isnull().sum()
miss = miss[miss > 0]
if len(miss):
    print(f'Columns with missing values: {len(miss)}')
    print(miss.head(10))
else:
    print('✓ No missing values in dataset')

FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/master_dataset.csv'

## 2. PM2.5 Distribution Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 2a. Histogram + KDE
ax = axes[0, 0]
ax.hist(df['pm25'].dropna(), bins=80, color='#4af0c8', alpha=0.6,
        edgecolor='none', density=True, label='Observed')
from scipy.stats import gaussian_kde
kde = gaussian_kde(df['pm25'].dropna())
x_range = np.linspace(df['pm25'].min(), df['pm25'].max(), 300)
ax.plot(x_range, kde(x_range), color='#c8f04a', linewidth=2, label='KDE')
ax.axvline(WHO_ANNUAL, color='#f0c84a', linestyle='--', linewidth=1.5, label=f'WHO Annual ({WHO_ANNUAL})')
ax.axvline(WHO_24H,    color='#f04a7a', linestyle='--', linewidth=1.5, label=f'WHO 24h ({WHO_24H})')
ax.axvline(df['pm25'].mean(), color='white', linestyle=':', linewidth=1.5,
           label=f'Mean ({df["pm25"].mean():.1f})')
ax.set_xlabel('PM2.5 (µg/m³)')
ax.set_ylabel('Density')
ax.set_title('PM2.5 Distribution — All Stations 2019–2023', color='#fff')
ax.legend(facecolor='#16161f', labelcolor='#aaa', edgecolor='#333', fontsize=8)

# 2b. Log-scale histogram
ax = axes[0, 1]
log_pm = np.log1p(df['pm25'].dropna())
ax.hist(log_pm, bins=60, color='#f04a7a', alpha=0.7, edgecolor='none', density=True)
mu, sigma = log_pm.mean(), log_pm.std()
x_log = np.linspace(log_pm.min(), log_pm.max(), 300)
ax.plot(x_log, stats.norm.pdf(x_log, mu, sigma), color='#c8f04a', linewidth=2,
        label=f'Normal fit (μ={mu:.2f}, σ={sigma:.2f})')
ax.set_xlabel('log(1 + PM2.5)')
ax.set_ylabel('Density')
ax.set_title('Log-Transformed PM2.5 (Normality Check)', color='#fff')
ax.legend(facecolor='#16161f', labelcolor='#aaa', edgecolor='#333', fontsize=8)

# 2c. Box plot by year
ax = axes[1, 0]
years = sorted(df['date'].dt.year.unique())
data_by_year = [df[df['date'].dt.year == y]['pm25'].dropna().values for y in years]
bp = ax.boxplot(data_by_year, positions=range(len(years)), widths=0.5,
                patch_artist=True, notch=False,
                boxprops=dict(facecolor='#4af0c8', alpha=0.5),
                medianprops=dict(color='white', linewidth=2),
                whiskerprops=dict(color='#4af0c8'),
                capprops=dict(color='#4af0c8'),
                flierprops=dict(marker='.', color='#4af0c8', alpha=0.2, markersize=2))
ax.set_xticks(range(len(years)))
ax.set_xticklabels(years)
ax.set_xlabel('Year')
ax.set_ylabel('PM2.5 (µg/m³)')
ax.set_title('Year-on-Year PM2.5 Distribution', color='#fff')
ax.axhline(WHO_ANNUAL, color='#f0c84a', linestyle='--', alpha=0.7, linewidth=1)
ax.axhline(WHO_24H,    color='#f04a7a', linestyle='--', alpha=0.7, linewidth=1)

# 2d. QQ plot
ax = axes[1, 1]
sample = df['pm25'].dropna().sample(min(3000, len(df)), random_state=42)
(osm, osr), (slope, intercept, r) = stats.probplot(np.log1p(sample), dist='norm')
ax.scatter(osm, osr, color='#4af0c8', alpha=0.3, s=5, label='log(PM2.5)')
ax.plot(osm, slope * np.array(osm) + intercept, color='#f04a7a', linewidth=2, label=f'R={r:.3f}')
ax.set_xlabel('Theoretical Quantiles')
ax.set_ylabel('Sample Quantiles')
ax.set_title('Q-Q Plot — log(PM2.5) vs Normal', color='#fff')
ax.legend(facecolor='#16161f', labelcolor='#aaa', edgecolor='#333')

plt.suptitle('PM2.5 Statistical Distribution Analysis — Karachi 2019–2023',
             color='#fff', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/03_pm25_distribution.png', dpi=150, bbox_inches='tight',
            facecolor='#0d0d14')
plt.show()
print('✓ Saved → outputs/03_pm25_distribution.png')

# Shapiro-Wilk on log-transformed
_, p_sw = stats.shapiro(np.log1p(sample))
skew    = stats.skew(df['pm25'].dropna())
kurt    = stats.kurtosis(df['pm25'].dropna())
print(f'\nDistribution Stats:')
print(f'  Skewness        : {skew:.3f} (>0 = right skewed)')
print(f'  Kurtosis        : {kurt:.3f}')
print(f'  Shapiro-Wilk p  : {p_sw:.4f} ({"normal" if p_sw > 0.05 else "non-normal"} after log-transform)')

## 3. Temporal Trends

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(18, 14))

MONTHS = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
DAYS   = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']

# 3a. Full time-series (30-day rolling)
ax = axes[0]
daily_mean = df.groupby('date')['pm25'].mean()
roll30     = daily_mean.rolling(30, center=True, min_periods=1).mean()

ax.fill_between(daily_mean.index, daily_mean.values,
                alpha=0.2, color='#4af0c8', label='Daily mean')
ax.plot(daily_mean.index, daily_mean.values,
        color='#4af0c8', alpha=0.4, linewidth=0.5)
ax.plot(roll30.index, roll30.values,
        color='#c8f04a', linewidth=2, label='30-day rolling mean')
ax.axhline(WHO_ANNUAL, color='#f0c84a', linestyle='--', linewidth=1.5,
           label=f'WHO Annual Guideline ({WHO_ANNUAL} µg/m³)')
ax.axhline(WHO_24H, color='#f04a7a', linestyle='--', linewidth=1.5,
           label=f'WHO 24h Guideline ({WHO_24H} µg/m³)')

# Shade monsoon periods
for year in df['date'].dt.year.unique():
    ax.axvspan(pd.Timestamp(f'{year}-07-01'), pd.Timestamp(f'{year}-10-31'),
               alpha=0.08, color='#4a7af0', label='Monsoon' if year == df['date'].dt.year.min() else '')

ax.set_ylabel('PM2.5 (µg/m³)')
ax.set_title('Daily PM2.5 Time Series — City-wide Mean (2019–2023)', color='#fff')
ax.legend(facecolor='#16161f', labelcolor='#aaa', edgecolor='#333', fontsize=8, ncol=3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)

# 3b. Monthly climatology
ax = axes[1]
monthly = df.groupby(df['date'].dt.month)['pm25'].agg(['mean','std','median'])
x = np.arange(1, 13)
ax.bar(x, monthly['mean'], color=PALETTE[:12], alpha=0.7, width=0.6)
ax.errorbar(x, monthly['mean'], yerr=monthly['std'],
            fmt='none', color='white', capsize=5, linewidth=1.5, alpha=0.7)
ax.plot(x, monthly['median'], color='#c8f04a', marker='D',
        markersize=6, linewidth=1.5, label='Median')
ax.set_xticks(x)
ax.set_xticklabels(MONTHS)
ax.set_ylabel('PM2.5 (µg/m³)')
ax.set_title('Monthly PM2.5 Climatology (Mean ± Std)', color='#fff')
ax.axhline(WHO_24H, color='#f04a7a', linestyle='--', linewidth=1, alpha=0.7)
ax.legend(facecolor='#16161f', labelcolor='#aaa', edgecolor='#333')

# Annotate peak months
peak_month = monthly['mean'].idxmax()
ax.annotate(f'Peak: {MONTHS[peak_month-1]}\n{monthly["mean"][peak_month]:.1f} µg/m³',
            xy=(peak_month, monthly['mean'][peak_month]),
            xytext=(peak_month + 0.8, monthly['mean'][peak_month] + 5),
            color='#f04a7a', fontsize=9,
            arrowprops=dict(arrowstyle='->', color='#f04a7a'))

# 3c. Day-of-week pattern
ax = axes[2]
dow = df.groupby(df['date'].dt.dayofweek)['pm25'].agg(['mean','std'])
colors_dow = ['#4af0c8'] * 5 + ['#f04a7a'] * 2  # weekday vs weekend
ax.bar(range(7), dow['mean'], color=colors_dow, alpha=0.75, width=0.6)
ax.errorbar(range(7), dow['mean'], yerr=dow['std'],
            fmt='none', color='white', capsize=5, linewidth=1.5, alpha=0.7)
ax.set_xticks(range(7))
ax.set_xticklabels(DAYS)
ax.set_ylabel('Mean PM2.5 (µg/m³)')
ax.set_title('Day-of-Week PM2.5 Pattern (Teal=Weekday, Red=Weekend)', color='#fff')

# Weekend effect
weekday_mean = dow['mean'][:5].mean()
weekend_mean = dow['mean'][5:].mean()
effect = ((weekday_mean - weekend_mean) / weekday_mean) * 100
ax.text(0.02, 0.92, f'Weekend effect: {effect:+.1f}% vs weekday',
        transform=ax.transAxes, color='#f0c84a', fontsize=9)

plt.suptitle('Temporal Analysis of PM2.5 — Karachi 2019–2023',
             color='#fff', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/03_temporal_trends.png', dpi=150, bbox_inches='tight',
            facecolor='#0d0d14')
plt.show()
print('✓ Saved → outputs/03_temporal_trends.png')

## 4. Seasonal Analysis

In [ ]:
season_map = {12:'Winter',1:'Winter',2:'Winter',
              3:'Pre-Monsoon',4:'Pre-Monsoon',5:'Pre-Monsoon',6:'Pre-Monsoon',
              7:'Monsoon',8:'Monsoon',9:'Monsoon',10:'Monsoon',
              11:'Post-Monsoon'}
df['season_label'] = df['date'].dt.month.map(season_map)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Violin plot by season
ax = axes[0]
season_order = ['Winter','Pre-Monsoon','Monsoon','Post-Monsoon']
season_colors = ['#4a7af0','#f0c84a','#4af0c8','#f07a4a']

for i, (season, color) in enumerate(zip(season_order, season_colors)):
    vals = df[df['season_label'] == season]['pm25'].dropna()
    parts = ax.violinplot(vals, positions=[i], widths=0.6,
                          showmeans=True, showmedians=True)
    for pc in parts['bodies']:
        pc.set_facecolor(color)
        pc.set_alpha(0.6)
    for partname in ['cmeans','cmedians','cbars','cmins','cmaxes']:
        if partname in parts:
            parts[partname].set_color('white')
    ax.text(i, vals.mean() + 2, f'{vals.mean():.1f}',
            ha='center', fontsize=9, color=color, fontweight='bold')

ax.set_xticks(range(4))
ax.set_xticklabels(season_order)
ax.set_ylabel('PM2.5 (µg/m³)')
ax.set_title('PM2.5 by Season — Violin Plot', color='#fff')
ax.axhline(WHO_24H, color='#f04a7a', linestyle='--', linewidth=1, alpha=0.7)

# Year × Season heatmap
ax = axes[1]
pivot = df.groupby([df['date'].dt.year, 'season_label'])['pm25'].mean().unstack()
pivot = pivot[season_order]  # enforce order
im = ax.imshow(pivot.values, aspect='auto', cmap='RdYlGn_r',
               vmin=pivot.values.min(), vmax=pivot.values.max())
plt.colorbar(im, ax=ax, label='Mean PM2.5 (µg/m³)', shrink=0.8)
ax.set_xticks(range(len(season_order)))
ax.set_xticklabels(season_order, rotation=15)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
ax.set_title('Year × Season PM2.5 Heatmap', color='#fff')
for i in range(len(pivot.index)):
    for j in range(len(season_order)):
        ax.text(j, i, f'{pivot.values[i, j]:.1f}',
                ha='center', va='center', fontsize=9, color='white', fontweight='bold')

plt.suptitle('Seasonal PM2.5 Patterns — Karachi', color='#fff', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/03_seasonal_analysis.png', dpi=150, bbox_inches='tight',
            facecolor='#0d0d14')
plt.show()
print('✓ Saved → outputs/03_seasonal_analysis.png')

print('\nSeasonal Summary:')
print(df.groupby('season_label')['pm25'].describe().round(1)[['mean','std','min','max']].loc[season_order])

## 5. Station-Level Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

station_stats = df.groupby('station')['pm25'].agg(['mean','std','median',
    lambda x: x.quantile(0.95)]).reset_index()
station_stats.columns = ['station','mean','std','median','p95']
station_stats = station_stats.sort_values('mean', ascending=False)

# Bar chart
ax = axes[0]
x  = np.arange(len(station_stats))
bars = ax.bar(x, station_stats['mean'], color=PALETTE[:len(station_stats)],
              alpha=0.8, width=0.6)
ax.errorbar(x, station_stats['mean'], yerr=station_stats['std'],
            fmt='none', color='white', capsize=5, linewidth=1.5)
ax.set_xticks(x)
ax.set_xticklabels([s.replace('_',' ') for s in station_stats['station']],
                   rotation=35, ha='right', fontsize=8)
ax.set_ylabel('Annual Mean PM2.5 (µg/m³)')
ax.set_title('PM2.5 by Station — Mean ± Std', color='#fff')
ax.axhline(WHO_ANNUAL, color='#f0c84a', linestyle='--', linewidth=1.5,
           label=f'WHO ({WHO_ANNUAL} µg/m³)')
ax.axhline(WHO_24H, color='#f04a7a', linestyle='--', linewidth=1.5,
           label=f'WHO 24h ({WHO_24H} µg/m³)')
for bar, val in zip(bars, station_stats['mean']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}', ha='center', fontsize=8, color='white')
ax.legend(facecolor='#16161f', labelcolor='#aaa', edgecolor='#333')

# Monthly line per station
ax = axes[1]
for i, station in enumerate(df['station'].unique()):
    sdf = df[df['station'] == station]
    monthly_s = sdf.groupby(sdf['date'].dt.month)['pm25'].mean()
    ax.plot(range(1, 13), monthly_s.values, color=PALETTE[i % len(PALETTE)],
            linewidth=1.8, marker='o', markersize=4,
            label=station.replace('_',' '))

MONTHS = ['J','F','M','A','M','J','J','A','S','O','N','D']
ax.set_xticks(range(1, 13))
ax.set_xticklabels(MONTHS)
ax.set_ylabel('Mean PM2.5 (µg/m³)')
ax.set_title('Monthly PM2.5 Profile by Station', color='#fff')
ax.legend(facecolor='#16161f', labelcolor='#aaa', edgecolor='#333',
          fontsize=7, ncol=2)
ax.axvspan(7, 10, alpha=0.1, color='#4af0c8')
ax.text(8.2, ax.get_ylim()[0] + 1, 'Monsoon', color='#4af0c8', fontsize=8)

plt.suptitle('Station-Level PM2.5 Comparison', color='#fff', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/03_station_comparison.png', dpi=150, bbox_inches='tight',
            facecolor='#0d0d14')
plt.show()
print('✓ Saved → outputs/03_station_comparison.png')
print()
print(station_stats.to_string(index=False))

## 6. Meteorological Correlations

In [ ]:
temp_col = 'temp_c' if 'temp_c' in df.columns else 'temperature_2m'
met_features = {
    temp_col       : 'Temperature (°C)',
    'wind_speed'   : 'Wind Speed (m/s)',
    'rh'           : 'Relative Humidity (%)',
}
# Add optional columns if present
for col, label in [('precip_mm','Precipitation (mm)'),
                   ('pressure_hpa','Pressure (hPa)'),
                   ('stagnation_index','Stagnation Index')]:
    if col in df.columns:
        met_features[col] = label

n = len(met_features)
cols_list = list(met_features.keys())
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, (col, label) in zip(axes, met_features.items()):
    valid = df[['pm25', col]].dropna()
    r, p  = stats.pearsonr(valid['pm25'], valid[col])

    sample = valid.sample(min(3000, len(valid)), random_state=42)
    ax.scatter(sample[col], sample['pm25'], alpha=0.15, s=4,
               color='#4af0c8', edgecolors='none')

    # Regression line
    m, b = np.polyfit(valid[col], valid['pm25'], 1)
    x_fit = np.linspace(valid[col].min(), valid[col].max(), 100)
    ax.plot(x_fit, m * x_fit + b, color='#f04a7a', linewidth=2)

    ax.set_xlabel(label)
    ax.set_ylabel('PM2.5 (µg/m³)')
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    ax.set_title(f'r = {r:.3f} {sig}', color='#fff')
    ax.text(0.05, 0.92, label, transform=ax.transAxes,
            color='#aaa', fontsize=9)

# Hide any unused axes
for ax in axes[n:]:
    ax.set_visible(False)

plt.suptitle('PM2.5 vs Meteorological Variables — Pearson Correlations',
             color='#fff', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/03_met_correlations.png', dpi=150, bbox_inches='tight',
            facecolor='#0d0d14')
plt.show()
print('✓ Saved → outputs/03_met_correlations.png')

print('\nCorrelation Table (PM2.5 vs meteorological variables):')
for col, label in met_features.items():
    valid = df[['pm25', col]].dropna()
    r, p  = stats.pearsonr(valid['pm25'], valid[col])
    sig   = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    print(f'  {label:<28} r={r:+.4f}  p={p:.4f}  {sig}')

## 7. Satellite Feature Analysis (AOD, AER-AI, VIIRS)

In [ ]:
sat_features = {}
for col, label in [('Optical_Depth_055','MODIS AOD 550nm'),
                   ('Optical_Depth_047','MODIS AOD 470nm'),
                   ('aer_ai','Sentinel-5P AER-AI'),
                   ('viirs_ntl','VIIRS Nighttime Light'),
                   ('no2','Sentinel-5P NO₂'),
                   ('co','Sentinel-5P CO')]:
    if col in df.columns:
        sat_features[col] = label

n = len(sat_features)
cols_sat = list(sat_features.keys())
ncols = 3
nrows = (n + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(18, 5 * nrows))
axes = axes.flatten() if n > 1 else [axes]

for ax, (col, label) in zip(axes, sat_features.items()):
    valid = df[['pm25', col]].dropna()
    r, p  = stats.pearsonr(valid['pm25'], valid[col])
    rs, _ = stats.spearmanr(valid['pm25'], valid[col])

    sample = valid.sample(min(3000, len(valid)), random_state=42)
    sc = ax.scatter(sample[col], sample['pm25'], alpha=0.2, s=5,
                    c=sample['pm25'], cmap='RdYlGn_r', edgecolors='none')

    m, b = np.polyfit(valid[col], valid['pm25'], 1)
    x_fit = np.linspace(valid[col].quantile(0.01), valid[col].quantile(0.99), 100)
    ax.plot(x_fit, m * x_fit + b, color='white', linewidth=2, linestyle='--')

    ax.set_xlabel(label)
    ax.set_ylabel('PM2.5 (µg/m³)')
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    ax.set_title(f'Pearson r={r:.3f} | Spearman ρ={rs:.3f} {sig}', color='#fff', fontsize=10)
    ax.text(0.05, 0.92, label, transform=ax.transAxes, color='#c8f04a', fontsize=9)

for ax in axes[n:]:
    ax.set_visible(False)

plt.suptitle('PM2.5 vs Satellite Remote Sensing Features',
             color='#fff', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/03_satellite_correlations.png', dpi=150, bbox_inches='tight',
            facecolor='#0d0d14')
plt.show()
print('✓ Saved → outputs/03_satellite_correlations.png')

print('\nSatellite Feature Correlations with PM2.5:')
for col, label in sat_features.items():
    valid = df[['pm25', col]].dropna()
    r, p  = stats.pearsonr(valid['pm25'], valid[col])
    rs, _ = stats.spearmanr(valid['pm25'], valid[col])
    sig   = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    print(f'  {label:<28} Pearson r={r:+.4f}  Spearman ρ={rs:+.4f}  {sig}')

## 8. Full Correlation Heatmap

In [ ]:
temp_col = 'temp_c' if 'temp_c' in df.columns else 'temperature_2m'

CORR_COLS = ['pm25']
for col in ['Optical_Depth_055','Optical_Depth_047','aer_ai','no2','so2','co',
            'viirs_ntl',temp_col,'wind_speed','rh','precip_mm','pressure_hpa',
            'stagnation_index','month','day_of_week','is_weekend']:
    if col in df.columns:
        CORR_COLS.append(col)

corr_df  = df[CORR_COLS].dropna()
corr_mat = corr_df.corr()

LABELS = {
    'pm25':'PM2.5','Optical_Depth_055':'AOD 550','Optical_Depth_047':'AOD 470',
    'aer_ai':'AER-AI','no2':'NO₂','so2':'SO₂','co':'CO','viirs_ntl':'NTL',
    'temp_c':'Temp','temperature_2m':'Temp','wind_speed':'Wind',
    'rh':'RH','precip_mm':'Precip','pressure_hpa':'Pressure',
    'stagnation_index':'Stagnation','month':'Month',
    'day_of_week':'DoW','is_weekend':'Weekend'
}
labels = [LABELS.get(c, c) for c in CORR_COLS]

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_mat, dtype=bool), k=1)

cmap = sns.diverging_palette(240, 10, as_cmap=True)
sns.heatmap(
    corr_mat, mask=mask, ax=ax,
    cmap=cmap, vmin=-1, vmax=1, center=0,
    annot=True, fmt='.2f', annot_kws={'size': 8, 'color': 'white'},
    linewidths=0.5, linecolor='#1a1a2a',
    xticklabels=labels, yticklabels=labels,
    square=True, cbar_kws={'shrink': 0.8, 'label': 'Pearson r'}
)
ax.set_title('Feature Correlation Matrix — PM2.5 + Predictors',
             color='#fff', fontsize=13, pad=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)

plt.tight_layout()
plt.savefig('outputs/03_correlation_heatmap.png', dpi=150, bbox_inches='tight',
            facecolor='#0d0d14')
plt.show()
print('✓ Saved → outputs/03_correlation_heatmap.png')

# Top correlations with PM2.5
pm25_corr = corr_mat['pm25'].drop('pm25').abs().sort_values(ascending=False)
print('\nTop 10 features by absolute correlation with PM2.5:')
for feat, val in pm25_corr.head(10).items():
    raw = corr_mat['pm25'][feat]
    print(f'  {LABELS.get(feat,feat):<20} |r| = {val:.4f}  (r={raw:+.4f})')

## 9. WHO Exceedance Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 9a. Exceedance days per year
ax = axes[0]
df['year'] = df['date'].dt.year
exceed_24h = df.groupby('year').apply(lambda x: (x['pm25'] > WHO_24H).sum())
exceed_ann = df.groupby('year').apply(lambda x: (x['pm25'] > WHO_ANNUAL).sum())
total_days  = df.groupby('year')['pm25'].count()

x = np.arange(len(exceed_24h))
w = 0.35
ax.bar(x - w/2, (exceed_ann / total_days * 100), w, label=f'> WHO Annual ({WHO_ANNUAL} µg/m³)',
       color='#f0c84a', alpha=0.8)
ax.bar(x + w/2, (exceed_24h / total_days * 100), w, label=f'> WHO 24h ({WHO_24H} µg/m³)',
       color='#f04a7a', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(exceed_24h.index)
ax.set_ylabel('% of Days Exceeding Guideline')
ax.set_title('WHO PM2.5 Guideline Exceedance by Year', color='#fff')
ax.legend(facecolor='#16161f', labelcolor='#aaa', edgecolor='#333')
ax.set_ylim(0, 110)
for i, (a, b_) in enumerate(zip(exceed_ann/total_days*100, exceed_24h/total_days*100)):
    ax.text(i - w/2, a + 1, f'{a:.0f}%', ha='center', fontsize=8, color='#f0c84a')
    ax.text(i + w/2, b_ + 1, f'{b_:.0f}%', ha='center', fontsize=8, color='#f04a7a')

# 9b. Exceedance by season
ax = axes[1]
season_exceed = df.groupby('season_label').apply(
    lambda x: pd.Series({
        'WHO Annual %': (x['pm25'] > WHO_ANNUAL).mean() * 100,
        'WHO 24h %'   : (x['pm25'] > WHO_24H).mean() * 100,
        'Mean PM2.5'  : x['pm25'].mean()
    })
).loc[['Winter','Pre-Monsoon','Monsoon','Post-Monsoon']]

x = np.arange(4)
ax.bar(x - w/2, season_exceed['WHO Annual %'], w, color='#f0c84a', alpha=0.8, label='> WHO Annual')
ax.bar(x + w/2, season_exceed['WHO 24h %'],    w, color='#f04a7a', alpha=0.8, label='> WHO 24h')
ax.set_xticks(x)
ax.set_xticklabels(['Winter','Pre-\nMonsoon','Monsoon','Post-\nMonsoon'])
ax.set_ylabel('% of Days Exceeding Guideline')
ax.set_title('WHO Exceedance by Season', color='#fff')
ax.legend(facecolor='#16161f', labelcolor='#aaa', edgecolor='#333')

plt.suptitle('WHO PM2.5 Guideline Exceedance Analysis — Karachi',
             color='#fff', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/03_who_exceedance.png', dpi=150, bbox_inches='tight',
            facecolor='#0d0d14')
plt.show()
print('✓ Saved → outputs/03_who_exceedance.png')

overall_24h = (df['pm25'] > WHO_24H).mean() * 100
overall_ann = (df['pm25'] > WHO_ANNUAL).mean() * 100
print(f'\nOverall exceedance: {overall_ann:.1f}% days > WHO annual | {overall_24h:.1f}% days > WHO 24h')
print(f'Annual mean PM2.5: {df["pm25"].mean():.1f} µg/m³ = {df["pm25"].mean()/WHO_ANNUAL:.1f}× WHO annual guideline')

## 10. Summary

In [ ]:
print('=' * 60)
print('✅ NOTEBOOK 03 COMPLETE — EDA')
print('=' * 60)
print()
print('Key findings:')
print(f'  • Dataset    : {df.shape[0]:,} rows, {df["station"].nunique()} stations, {df["date"].dt.year.nunique()} years')
print(f'  • Mean PM2.5 : {df["pm25"].mean():.1f} µg/m³ ({df["pm25"].mean()/WHO_ANNUAL:.1f}× WHO annual)')
print(f'  • Peak month : {["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"][df.groupby(df["date"].dt.month)["pm25"].mean().idxmax()-1]}')
print(f'  • WHO 24h exceedance: {(df["pm25"] > WHO_24H).mean()*100:.1f}% of all days')
print()
print('Saved outputs:')
for f in ['outputs/03_pm25_distribution.png',
          'outputs/03_temporal_trends.png',
          'outputs/03_seasonal_analysis.png',
          'outputs/03_station_comparison.png',
          'outputs/03_met_correlations.png',
          'outputs/03_satellite_correlations.png',
          'outputs/03_correlation_heatmap.png',
          'outputs/03_who_exceedance.png']:
    exists = '✓' if Path(f).exists() else '✗'
    print(f'  {exists} {f}')
print()
print('NEXT: Run 04_feature_selection.ipynb')